In [1]:
# [1] 필요한 라이브러리 불러오기 및 .env 파일에서 API Key 로딩

from dotenv import load_dotenv
import os

# .env 파일에서 환경변수 불러오기
load_dotenv()

# KAMIS-like API key 변수명
API_KEY = os.getenv("KAMIS_API_KEY")

# 확인용 출력 (주의: 실제 배포 시에는 노출 금지)
print("API Key Loaded:", "✅" if API_KEY else "❌ 오류: API Key 없음")



API Key Loaded: ✅


In [9]:
# [11] 전체 품목 중 '배추', '쌀', '양파', '상추', '사과' 포함된 중분류 이름/코드 자동 추출

import requests
import xml.etree.ElementTree as ET

def auto_find_middle_codes(api_key, keywords, chunk_size=1000, max_rows=13249):
    base_url = "http://211.237.50.150:7080/openapi"
    grid_id = "Grid_20141221000000000120_1"
    
    found = {}
    
    for start in range(1, max_rows + 1, chunk_size):
        end = min(start + chunk_size - 1, max_rows)
        url = f"{base_url}/{api_key}/xml/{grid_id}/{start}/{end}"

        try:
            response = requests.get(url)
            response.raise_for_status()
            root = ET.fromstring(response.text)
            rows = root.findall(".//row")
            
            for row in rows:
                name = row.findtext("PRDLST_NM")
                code = row.findtext("PRDLST_CD")
                for keyword in keywords:
                    if keyword in name and keyword not in found:
                        found[keyword] = {"품목명": name, "코드": code}
                        print(f"✅ {keyword} → {name} (코드: {code})")
                if len(found) == len(keywords):
                    break
        except Exception as e:
            print(f"❌ 오류: {e}")
            break
        
        if len(found) == len(keywords):
            break

    return found

# ✅ 실행
keywords = ["쌀", "배추", "양파", "상추", "사과"]
code_dict = auto_find_middle_codes(API_KEY, keywords)

print("\n📌 최종 중분류 코드 매핑 결과:")
for k, v in code_dict.items():
    print(f"{k}: {v['품목명']} (코드: {v['코드']})")


✅ 사과 → 사과 (코드: 19I9)
✅ 양파 → 양파 (코드: 1201)
✅ 배추 → 양배추 (코드: 1004)
✅ 상추 → 상추 (코드: 1005)
✅ 쌀 → 쌀 (코드: 0103)

📌 최종 중분류 코드 매핑 결과:
사과: 사과 (코드: 19I9)
양파: 양파 (코드: 1201)
배추: 양배추 (코드: 1004)
상추: 상추 (코드: 1005)
쌀: 쌀 (코드: 0103)


In [14]:
# [13] 소분류(SPCIES_NM) 중 '배추' 포함된 항목 찾기

def find_species_names_with(keyword: str, api_key: str, start=1, end=13249):
    url = f"http://211.237.50.150:7080/openapi/{api_key}/xml/Grid_20141221000000000120_1/{start}/{end}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        root = ET.fromstring(response.text)

        rows = root.findall(".//row")

        print(f"📋 소분류 품종명(SPCIES_NM) 중 '{keyword}' 포함된 항목:\n")
        count = 0
        for row in rows:
            species = row.find("SPCIES_NM")
            code = row.find("SPCIES_CD")
            prd = row.find("PRDLST_NM")
            if species is not None and keyword in species.text:
                print(f"- {species.text} (품목: {prd.text}, 소분류코드: {code.text})")
                count += 1
        print(f"\n총 {count}개 발견됨")
    except Exception as e:
        print("❌ 오류:", e)

# ✅ 실행
find_species_names_with("", API_KEY)


📋 소분류 품종명(SPCIES_NM) 중 '' 포함된 항목:


총 0개 발견됨


In [15]:
# [1] 수집 설정 셀: 기간, 품목 코드 매핑, 날짜 생성

import os
from datetime import datetime, timedelta
import pandas as pd

# ✅ API 키 로딩
API_KEY = os.getenv("KAMIS_API_KEY")  # .env에서 자동 불러옴

# ✅ 수집할 중분류 품목 코드 매핑 (배추 제외)
ITEM_CODES = {
    "쌀": "0103",
    "양파": "1201",
    "상추": "1005",
    "사과": "19I9"
}

# ✅ 수집할 날짜 리스트 생성: 2015-01-01 ~ 2024-12-01, 매월 1일
def generate_monthly_dates(start_year=2015, end_year=2024):
    date_list = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            date_str = f"{year}{month:02}01"
            date_list.append(date_str)
    return date_list

DATES = generate_monthly_dates()

# ✅ 검증 출력
print(f"📅 수집 날짜 수: {len(DATES)}개")
print(f"🎯 수집 품목: {list(ITEM_CODES.keys())}")
print(f"🔑 API 키: {'OK' if API_KEY else '❌ 없음'}")


📅 수집 날짜 수: 120개
🎯 수집 품목: ['쌀', '양파', '상추', '사과']
🔑 API 키: OK


In [16]:
# [2] 실시간 경락 정보 수집 함수 (품목명 + 날짜 입력 → 리스트 반환)

import requests
import xml.etree.ElementTree as ET

def fetch_price_data(item_name, item_code, saledate, api_key=API_KEY, start=1, end=1000):
    """
    품목코드와 경락일자 기준으로 도매시장 경락 데이터 수집
    - PRDLST_CD: 중분류 품목 코드
    - SALEDATE: YYYYMMDD 형식
    """
    base_url = "http://211.237.50.150:7080/openapi"
    grid_id = "Grid_20240625000000000654_1"
    url = f"{base_url}/{api_key}/xml/{grid_id}/{start}/{end}"
    params = {
        "SALEDATE": saledate,
        "PRDLST_CD": item_code,
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        root = ET.fromstring(response.text)
        rows = root.findall(".//row")
        
        results = []
        for row in rows:
            result = {
                "날짜": saledate,
                "품목명": item_name,
                "도매시장": row.findtext("WHSALNAME"),
                "법인명": row.findtext("CMPNAME"),
                "산지": row.findtext("SANNAME"),
                "도매단가": row.findtext("COST"),
                "거래량": row.findtext("QTY"),
                "규격": row.findtext("STD"),
                "입찰시각": row.findtext("SBIDTIME")
            }
            results.append(result)
        return results
    
    except Exception as e:
        print(f"❌ API 요청 오류: {saledate} / {item_name} →", e)
        return []

# ✅ 검증: 2020년 5월 1일, "쌀" 데이터 요청 테스트
test_data = fetch_price_data("쌀", "0103", "20200501")
print(f"📦 수집된 데이터 수: {len(test_data)}")
if test_data:
    print("예시 항목:", test_data[0])


📦 수집된 데이터 수: 0
